## Setup

Make sure you've installed dependencies from the project root:

```bash
pip install -r requirements.txt
```

> **Note:** This project uses **h3 v4.x**, which introduced significant API changes from v3 (e.g., `polyfill` → `geo_to_cells`, `h3_to_geo_boundary` → `cell_to_boundary`). Verify your version below.


In [ ]:
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

print(f"h3 version:        {h3.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"pandas version:    {pd.__version__}")

---

## Part 1: Convert Any Geography to H3 Hexagons

The first building block is a reusable function that takes any GeoDataFrame (counties, cities, census tracts, etc.) and returns a GeoDataFrame of H3 hexagons that tile the input area.

The function handles:

- CRS reprojection to WGS84
- Dissolving multi-feature inputs into a single geometry
- Converting each H3 cell ID into a Shapely polygon


In [ ]:
resolution = 7


def geodataframe_to_h3(gdf, resolution=resolution):
    """
    Generate individual H3 hexagons as separate features from an input geodataframe.

    Parameters:
        gdf (GeoDataFrame): Input geography (any CRS — will be reprojected to WGS84).
        resolution (int): H3 resolution level (0-15). Default is 7.

    Returns:
        GeoDataFrame: One row per hexagon, with columns for h3_id, geometry, and resolution.
    """

    # Convert to WGS84 if needed
    if gdf.crs != 'EPSG:4326':
        gdf = gdf.to_crs(epsg=4326)

    # Get the geometry (dissolve if multiple features)
    if len(gdf) > 1:
        geometry = gdf.dissolve().geometry.iloc[0]
    else:
        geometry = gdf.geometry.iloc[0]

    # Convert to H3 cells
    cells = h3.geo_to_cells(geometry, res=resolution)

    print(f"Generated {len(cells):,} H3 cells at resolution {resolution}")

    # Convert each cell to individual polygon features
    hex_polygons = []
    hex_ids = []

    for cell in cells:
        boundary = h3.cell_to_boundary(cell)
        coords = [(lng, lat) for lat, lng in boundary]
        polygon = Polygon(coords)
        hex_polygons.append(polygon)
        hex_ids.append(cell)

    hex_gdf = gpd.GeoDataFrame({
        'h3_id': hex_ids,
        'geometry': hex_polygons,
        'resolution': resolution
    }, crs='EPSG:4326')

    return hex_gdf

### Example usage

Read in a geography of interest and generate the hexagonal grid.

> **Try it yourself:** Swap in your own shapefile, census tracts from [TIGER/Line](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html), a city boundary, or any polygon layer. Experiment with different `resolution` values to see how the hexagon density changes.


In [ ]:
# Read in your geography of interest
metro_29 = gpd.read_file('../data/jurisdictions/ARC_29_counties.geojson')

# Generate the hexagonal grid
hex_gdf = geodataframe_to_h3(metro_29, resolution=7)

hex_gdf.head()

In [ ]:
# Visualize interactively — hex grid with input boundary overlaid
m = hex_gdf.explore(tooltip=False, popup=False,
                    highlight=False, tiles='CartoDB positron')
metro_29.explore(m=m, color='black', style_kwds={
                 'fillOpacity': 0, 'weight': 2}, tooltip=False, popup=False, highlight=False)
m

In [ ]:
# Export to GeoJSON
hex_gdf.to_file("../output/atl_29_Res7.geojson", driver="GeoJSON")
print("\u2713 Exported hex grid to output/atl_29_Res7.geojson")

---

## Part 2: Building the Transportation Impact Layer

Once you have a hexagonal grid, the real analytical power comes from **spatially joining data to it**. This section overlays GDOT transportation project lines onto the hex grid to create an "impact layer" showing where investment is concentrated.

### The problem

Transportation projects are lines (roads, corridors, transit routes) that often span multiple hexagons. Simply counting which hexagon a project's centroid falls in would be misleading — a 20-mile highway project would only register in one cell. Instead, we need to **split each project line by the hexagons it passes through** and allocate its cost proportionally.

### The approach

1. Filter to active projects (pre-construction and under-construction)
2. Project both layers to a planar CRS (`EPSG:3857`) for accurate length calculations
3. Use `gpd.overlay()` to intersect project lines with hexagons — clipping each line to hexagon boundaries
4. Calculate each clipped segment's proportion of the total project length
5. Allocate each project's cost estimate proportionally
6. Aggregate by hexagon to get total project counts and allocated investment per cell


In [ ]:
def allocate_projects_by_status(projects_gdf, hexagons_gdf):
    """
    Allocate transportation projects to hexagons proportionally by length.
    Separates pre-construction and under-construction projects.

    Parameters:
        projects_gdf (GeoDataFrame): GDOT project lines with 'Status' and 'Cost_estimate' columns.
        hexagons_gdf (GeoDataFrame): H3 hexagon grid.

    Returns:
        GeoDataFrame: Hexagons with aggregated project counts and allocated costs by status.
    """
    # Filter to active projects only
    active_projects = projects_gdf[
        projects_gdf['Status'].isin(['PRE-CONSTRUCTION', 'UNDER-CONSTRUCTION'])
    ].copy()

    if len(active_projects) == 0:
        print("No active projects found!")
        return hexagons_gdf

    # Project to planar CRS for accurate length calculations
    if active_projects.crs.is_geographic:
        projected_crs = 'EPSG:3857'
        projects_proj = active_projects.to_crs(projected_crs)
        hexagons_proj = hexagons_gdf.to_crs(projected_crs)
    else:
        projects_proj = active_projects.copy()
        hexagons_proj = hexagons_gdf.copy()

    # Add identifiers to preserve after overlay
    projects_proj['orig_project_idx'] = projects_proj.index
    hexagons_proj['hex_idx'] = hexagons_proj.index

    # Calculate original project lengths BEFORE overlay
    projects_proj['total_length'] = projects_proj.geometry.length

    # Perform overlay — clips each project line to hexagon boundaries
    overlays = gpd.overlay(projects_proj, hexagons_proj, how='intersection')

    # Calculate intersection lengths and proportional cost allocation
    overlays['intersection_length'] = overlays.geometry.length
    overlays['proportion'] = overlays['intersection_length'] / \
        overlays['total_length']
    overlays['allocated_cost'] = overlays['Cost_estimate'] * \
        overlays['proportion']

    # Separate by status and aggregate per hexagon
    pre_const = overlays[overlays['Status'] == 'PRE-CONSTRUCTION']
    under_const = overlays[overlays['Status'] == 'UNDER-CONSTRUCTION']

    if len(pre_const) > 0:
        pre_summary = pre_const.groupby('hex_idx').agg({
            'orig_project_idx': 'count',
            'allocated_cost': 'sum'
        }).rename(columns={
            'orig_project_idx': 'project_count_pre',
            'allocated_cost': 'allocated_cost_pre'
        })
    else:
        pre_summary = pd.DataFrame(
            columns=['project_count_pre', 'allocated_cost_pre'])

    if len(under_const) > 0:
        uc_summary = under_const.groupby('hex_idx').agg({
            'orig_project_idx': 'count',
            'allocated_cost': 'sum'
        }).rename(columns={
            'orig_project_idx': 'project_count_uc',
            'allocated_cost': 'allocated_cost_uc'
        })
    else:
        uc_summary = pd.DataFrame(
            columns=['project_count_uc', 'allocated_cost_uc'])

    # Join aggregated data back to original hexagons
    result = hexagons_gdf.copy()
    result['hex_idx'] = result.index

    if len(pre_summary) > 0:
        result = result.merge(pre_summary, left_on='hex_idx',
                              right_index=True, how='left')
    else:
        result['project_count_pre'] = 0
        result['allocated_cost_pre'] = 0.0

    if len(uc_summary) > 0:
        result = result.merge(uc_summary, left_on='hex_idx',
                              right_index=True, how='left')
    else:
        result['project_count_uc'] = 0
        result['allocated_cost_uc'] = 0.0

    # Fill NaN values for hexagons with no projects
    result['project_count_pre'] = result['project_count_pre'].fillna(
        0).astype(int)
    result['allocated_cost_pre'] = result['allocated_cost_pre'].fillna(0.0)
    result['project_count_uc'] = result['project_count_uc'].fillna(
        0).astype(int)
    result['allocated_cost_uc'] = result['allocated_cost_uc'].fillna(0.0)

    result = result.drop('hex_idx', axis=1)

    return result

### Load the data


In [ ]:
# Read in GDOT project data and the hex grid generated in Part 1
dot_projects = gpd.read_file('../data/projects/statewide_projects.geojson')
dot_projects = dot_projects[dot_projects['Status'] != 'UNKNOWN']

hex_layer = gpd.read_file('../output/atl_29_Res7.geojson')

print(f"Projects loaded: {len(dot_projects):,}")
print(f"Hexagons loaded: {len(hex_layer):,}")

### Build the impact layer


In [ ]:
result_hexagons = allocate_projects_by_status(dot_projects, hex_layer)

result_hexagons.head(3)

### Visualize the results

Filter to hexagons with nonzero under-construction investment and display as a choropleth.


In [ ]:
result_hexagons[result_hexagons['allocated_cost_uc'] > 0].explore(
    column='allocated_cost_uc',
    cmap='YlOrRd',
    scheme='FisherJenks',
    k=5,
    legend=False,
    highlight=False,
    tiles='CartoDB positron'
)

### Export


In [ ]:
result_hexagons.to_file('../output/impact_layer.geojson', driver='GeoJSON')
print("\u2713 Exported impact layer to output/impact_layer.geojson")